In [1]:
# # Kernel olarak nndet_venv seçin
# # veya başa ekleyin:
# import sys
# sys.path.insert(0, '/Users/ramazanpolat/Desktop/datasets/abdomen/nnDetection')
# sys.path.insert(0, '/Users/ramazanpolat/Desktop/datasets/abdomen/nndet_venv/lib/python3.9/site-packages')


## 1. Ortam ve nnDetection Kontrolü

In [2]:
import sys

!{sys.executable} -m pip install pydicom
!{sys.executable} -m pip install monai
!{sys.executable} -m pip install SimpleITK

print("Gerekli kütüphaneler başarıyla yüklendi.")

Gerekli kütüphaneler başarıyla yüklendi.


In [1]:
import os
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')

print(f"Ortam: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Yerel'}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive bağlandı.")

Ortam: Yerel


In [2]:
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Tuple

import sys, shutil, subprocess


In [3]:
import os, sys, shutil, subprocess, json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List

# ── Ortam bazlı yol konfigürasyonu ────────────────────────────────────────
if IS_KAGGLE:
    KAGGLE_INPUT = Path('/kaggle/input/datasets/ramazan2020/abdomen-acikveri')
    DATA_ROOT    = KAGGLE_INPUT
    EGITIM_DIR   = DATA_ROOT / "Egitim Verisi"
    SPLIT_DIR    = DATA_ROOT / "outputs/splits"
    CODE         = DATA_ROOT
    DET_DATA_DIR = DATA_ROOT / "det_data"
    NND_ROOT     = Path('/kaggle/working/nndet')
    NIFTI_DIR    = NND_ROOT / "nifti"

elif IS_COLAB:
    DRIVE_BASE   = Path('/content/drive/MyDrive/abdomen')
    DATA_ROOT    = DRIVE_BASE
    EGITIM_DIR   = DATA_ROOT / "Egitim Verisi"
    SPLIT_DIR    = DRIVE_BASE / "outputs/splits"
    CODE         = DRIVE_BASE
    DET_DATA_DIR = DRIVE_BASE / "outputs/det_data"
    NND_ROOT     = DRIVE_BASE / "outputs/nndet"
    NIFTI_DIR    = Path('/content/nndet/nifti')   # Colab yerel disk (hızlı I/O)

else:
    from dotenv import load_dotenv
    load_dotenv()
    DATA_ROOT    = Path(os.environ.get("TR_ABDOMEN_BASE", "."))
    PROJE        = Path(os.environ.get("TR_ABDOMEN_PROJE", "."))
    CODE         = Path(os.environ.get("TR_ABDOMEN_CODE",  "."))
    EGITIM_DIR   = DATA_ROOT / "Egitim Verisi"
    SPLIT_DIR    = PROJE / "outputs/splits"
    DET_DATA_DIR = PROJE / "outputs/det_data"
    NND_ROOT     = PROJE / "outputs/nndet"
    NIFTI_DIR    = NND_ROOT / "nifti"

if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

SUPER_CLASSES: List[str] = [
    "acute_cholecystitis",        # 0 → label 1
    "kidney_ureter_stone",        # 1 → label 2
    "acute_pancreatitis",         # 2 → label 3
    "aortic_aneurysm_dissection", # 3 → label 4
    "acute_appendicitis",         # 4 → label 5
    "acute_diverticulitis",       # 5 → label 6
]

# ── nnU-Net v2 yolları ─────────────────────────────────────────────────────
# Dataset dizini: {nnUNet_raw}/Dataset100_Abdomen/imagesTr|labelsTr
NNUNET_RAW          = NND_ROOT / "nnunet_raw"
NNUNET_PREPROCESSED = NND_ROOT / "nnunet_preprocessed"
NNUNET_RESULTS      = NND_ROOT / "nnunet_results"

DATASET_ID   = 100
DATASET_NAME = "Abdomen"
DATASET_DIR  = NNUNET_RAW / f"Dataset{DATASET_ID}_{DATASET_NAME}"

os.environ["nnUNet_raw"]          = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"]      = str(NNUNET_RESULTS)
os.environ["OMP_NUM_THREADS"]     = "1"

for p in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS,
          DATASET_DIR / "imagesTr", DATASET_DIR / "labelsTr", NIFTI_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def load_fold(fold: int, split: str) -> List[int]:
    p = SPLIT_DIR / f"fold{fold}_{split}.csv"
    return pd.read_csv(p)["Case Number"].astype(int).tolist()

fold = 0
print(f"nnUNet_raw         : {os.environ['nnUNet_raw']}")
print(f"nnUNet_preprocessed: {os.environ['nnUNet_preprocessed']}")
print(f"nnUNet_results     : {os.environ['nnUNet_results']}")
print(f"DATASET_DIR        : {DATASET_DIR}")
print(f"NIFTI_DIR          : {NIFTI_DIR}")
print(f"EGITIM_DIR         : {EGITIM_DIR}  → var? {EGITIM_DIR.exists()}")


nnUNet_raw         : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_raw
nnUNet_preprocessed: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_preprocessed
nnUNet_results     : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_results
DATASET_DIR        : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_raw/Dataset100_Abdomen
NIFTI_DIR          : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nifti
EGITIM_DIR         : /Users/ramazanpolat/Desktop/datasets/abdomen/Egitim Verisi  → var? True


### nnU-Net v2 Kurulumu

nnDetection yerine **nnU-Net v2** kullanıyoruz:
- `pip install nnunetv2` → bitti, CUDA derleme yok
- Semantic segmentation ile eğitim → predicted mask'lerden connected-component bbox çıkarımı
- Python 3.12 / PyTorch 2.x tam uyumlu

In [6]:
!pip install nnunetv2 scipy


  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


In [4]:
import importlib

print("nnU-Net v2 + scipy kuruluyor...")
_r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "nnunetv2", "scipy", "-q"],
    capture_output=True, text=True
)
if _r.returncode != 0:
    print("✗ Kurulum hatası:"); print(_r.stderr[-800:])
else:
    importlib.invalidate_caches()
    try:
        import nnunetv2
    except ImportError as e:
        print(f"✗ import hatası: {e}")

# Komut yollarını doğrula
import shutil as _sh, sysconfig as _sc
for _cmd in ["nnUNetv2_plan_and_preprocess", "nnUNetv2_train", "nnUNetv2_predict"]:
    _p = _sh.which(_cmd) or str(Path(_sc.get_path("scripts")) / _cmd)
    print(f"  {_cmd}: {_p}  (var={Path(_p).exists()})")


nnU-Net v2 + scipy kuruluyor...
  nnUNetv2_plan_and_preprocess: /Users/ramazanpolat/Desktop/datasets/abdomen/.venv/bin/nnUNetv2_plan_and_preprocess  (var=True)
  nnUNetv2_train: /Users/ramazanpolat/Desktop/datasets/abdomen/.venv/bin/nnUNetv2_train  (var=True)
  nnUNetv2_predict: /Users/ramazanpolat/Desktop/datasets/abdomen/.venv/bin/nnUNetv2_predict  (var=True)


## 2. DICOM Seri → NIfTI Hacim

In [8]:
import sys
!{sys.executable} -m pip install SimpleITK --only-binary=:all:

In [5]:
import SimpleITK as sitk
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

fold_cases = sorted(set(load_fold(fold, "train") + load_fold(fold, "val")))
print(f"Fold {fold}: {len(fold_cases)} vaka NIfTI'ye dönüştürülecek")
print(f"DICOM kaynak: {EGITIM_DIR}")
print(f"NIfTI hedef : {NIFTI_DIR}")

def dicom_to_nifti(case_dir: Path, out_path: Path) -> str:
    if out_path.exists():
        return "skip"
    reader = sitk.ImageSeriesReader()
    names = reader.GetGDCMSeriesFileNames(str(case_dir))
    if not names:
        return "no_dcm"
    reader.SetFileNames(names)
    try:
        img = reader.Execute()
        sitk.WriteImage(img, str(out_path))
        return "ok"
    except Exception as e:
        return f"err:{e}"

def _conv(case: int):
    cd = EGITIM_DIR / str(case)
    op = NIFTI_DIR / f"case_{case:05d}_0000.nii.gz"
    return case, dicom_to_nifti(cd, op)

with ThreadPoolExecutor(max_workers=4) as ex:
    results = list(tqdm(ex.map(_conv, fold_cases), total=len(fold_cases), desc="DICOM→NIfTI"))

ok   = sum(1 for _, s in results if s == "ok")
skip = sum(1 for _, s in results if s == "skip")
errs = [(c, s) for c, s in results if s not in ("ok", "skip")]
print(f"\n✓ Yeni: {ok}, ⏭️  Atlandı: {skip}, ❌ Hata: {len(errs)}")
if errs:
    print("İlk 5 hata:", errs[:5])

Fold 0: 554 vaka NIfTI'ye dönüştürülecek
DICOM kaynak: /Users/ramazanpolat/Desktop/datasets/abdomen/Egitim Verisi
NIfTI hedef : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nifti


DICOM→NIfTI: 100%|██████████| 554/554 [00:00<00:00, 3157.36it/s]


✓ Yeni: 0, ⏭️  Atlandı: 553, ❌ Hata: 1
İlk 5 hata: [(20317, 'err:Exception thrown in SimpleITK ImageSeriesReader_Execute: /Users/ec2-user/actions-runner/_work/SimpleITK/SimpleITK/bld/ITK-prefix/include/ITK-5.4/itkImageFileReader.hxx:338:\nImageIO returns IO region that does not fully contain the requested region. Requested region: ImageRegion (0x173d22ca8)\n  Dimension: 3\n  Index: [0, 0, 0]\n  Size: [608, 512, 1]\nStreamableRegion region: ImageRegion (0x173d22ce0)\n  Dimension: 3\n  Index: [0, 0, 0]\n  Size: [512, 512, 1]\n')]


## 3. 2B BBox → 3B Kutu Yükseltme

`Bilgi.xlsx`'teki tüm annotasyonlar 2B. 3B detection için ardışık kesit BBox'larını birleştirip 3B kutu üretiyoruz:
- Aynı vaka + aynı sınıf
- Ardışık veya yakın kesit (Δz ≤ 2)
- 2B IoU ≥ 0.3 → aynı lezyon

In [6]:
import numpy as np
import pandas as pd

import pydicom

def _2d_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2-ix1), max(0.0, iy2-iy1)
    inter = iw * ih
    ua = (ax2-ax1)*(ay2-ay1); ub = (bx2-bx1)*(by2-by1)
    return inter / max(ua + ub - inter, 1e-6)

def case_image_id_to_z(case):
    cd = EGITIM_DIR / str(case)
    files = sorted(
        [p for p in cd.glob("*.dcm") if not p.stem.startswith(".")],
        key=lambda p: int(p.stem)
    )
    positions = []
    for p in files:
        ds = pydicom.dcmread(str(p), stop_before_pixels=True)
        zpos = float(getattr(ds, 'ImagePositionPatient', [0,0,int(p.stem)])[2])
        positions.append((int(p.stem), zpos))
    positions.sort(key=lambda x: x[1])
    return {img_id: idx for idx, (img_id, _) in enumerate(positions)}

def lift_2d_to_3d(manifest, case, delta_z=2, iou_th=0.3):
    z_map = case_image_id_to_z(case)
    sub = manifest[(manifest['case'] == case) &
                   (manifest['bboxes'].fillna('').astype(str) != '')]
    items = []
    for _, row in sub.iterrows():
        z = z_map.get(int(row['image_id']))
        if z is None: continue
        for bb_str in str(row['bboxes']).split('|'):
            parts = bb_str.split(',')
            if len(parts) < 5: continue
            sid = int(parts[0]); x1,y1,x2,y2 = map(int, parts[1:5])
            items.append((sid, x1, y1, x2, y2, z))
    boxes_3d = []
    for sid in set(it[0] for it in items):
        cls_items = sorted([it for it in items if it[0] == sid], key=lambda x: x[5])
        used = set()
        for i, it in enumerate(cls_items):
            if i in used: continue
            grp = [it]; used.add(i)
            for j in range(i+1, len(cls_items)):
                if j in used: continue
                last = grp[-1]
                if cls_items[j][5] - last[5] <= delta_z:
                    if _2d_iou(last[1:5], cls_items[j][1:5]) >= iou_th:
                        grp.append(cls_items[j]); used.add(j)
                else: break
            xs1 = min(g[1] for g in grp); ys1 = min(g[2] for g in grp)
            xs2 = max(g[3] for g in grp); ys2 = max(g[4] for g in grp)
            zs1 = min(g[5] for g in grp); zs2 = max(g[5] for g in grp)
            boxes_3d.append({"class": sid, "x1": xs1, "y1": ys1, "z1": zs1,
                             "x2": xs2, "y2": ys2, "z2": zs2,
                             "n_slices": len(grp)})
    return boxes_3d


In [11]:
fold = 0
fold_cases = sorted(set(load_fold(fold, "train") + load_fold(fold, "val")))
print(f"Fold {fold}: {len(fold_cases)} vaka dönüştürülecek")

Fold 0: 554 vaka dönüştürülecek


In [7]:

manifest = pd.read_csv(SPLIT_DIR / "manifest.csv")
demo_case = fold_cases[0]
demo_boxes = lift_2d_to_3d(manifest, demo_case)
print(f"Vaka {demo_case}: {len(demo_boxes)} adet 3B kutu")
for b in demo_boxes[:5]:
    print(f"  sınıf {b['class']} ({SUPER_CLASSES[b['class']]}): "
          f"({b['x1']},{b['y1']},{b['z1']}) → ({b['x2']},{b['y2']},{b['z2']}), {b['n_slices']} kesit")

Vaka 20001: 1 adet 3B kutu
  sınıf 1 (kidney_ureter_stone): (251,290,11) → (262,302,12), 2 kesit


## 4. nnDetection Veri Formatı (Task100_Abdomen)

In [8]:
import json
import numpy as np
import SimpleITK as sitk

train_cases = load_fold(fold, "train")
val_cases   = load_fold(fold, "val")
all_cases   = train_cases + val_cases

def link_or_copy(src, dst):
    if dst.exists(): return
    try: os.symlink(src, dst)
    except (OSError, NotImplementedError): shutil.copy2(src, dst)

def prep_case_nnunet(case: int) -> str:
    """
    nnU-Net v2 için semantic segmentation maskesi oluşturur.
    Instance ID'ler yerine SINIF ID'leri (1-6) kullanılır.
    """
    src_nii = NIFTI_DIR / f"case_{case:05d}_0000.nii.gz"
    if not src_nii.exists():
        return f"missing:{case}"

    # Görüntüyü imagesTr'e bağla / kopyala
    dst_img = DATASET_DIR / "imagesTr" / f"case_{case:05d}_0000.nii.gz"
    link_or_copy(src_nii, dst_img)

    # Semantic maske zaten varsa atla
    dst_lbl = DATASET_DIR / "labelsTr" / f"case_{case:05d}.nii.gz"
    if dst_lbl.exists():
        return "skip"

    # 3B kutuları al, semantic maske oluştur
    boxes_3d = lift_2d_to_3d(manifest, case)
    img_itk  = sitk.ReadImage(str(src_nii))
    img_arr  = sitk.GetArrayFromImage(img_itk)     # [Z, Y, X]
    mask_arr = np.zeros(img_arr.shape, dtype=np.uint8)

    for b in boxes_3d:
        label = int(b["class"]) + 1               # 0-indexed class → 1-indexed label
        z1 = int(b["z1"]); z2 = min(int(b["z2"]) + 1, mask_arr.shape[0])
        y1 = int(b["y1"]); y2 = min(int(b["y2"]) + 1, mask_arr.shape[1])
        x1 = int(b["x1"]); x2 = min(int(b["x2"]) + 1, mask_arr.shape[2])
        # Birden fazla sınıf çakışırsa sonraki sınıf kazanır (nadir)
        mask_arr[z1:z2, y1:y2, x1:x2] = label

    mask_itk = sitk.GetImageFromArray(mask_arr)
    mask_itk.CopyInformation(img_itk)
    sitk.WriteImage(mask_itk, str(dst_lbl))
    return "ok"

manifest = pd.read_csv(SPLIT_DIR / "manifest.csv")
from tqdm import tqdm

results = [prep_case_nnunet(c) for c in tqdm(all_cases, desc="prep")]
ok   = results.count("ok")
skip = results.count("skip")
miss = [r for r in results if r.startswith("missing")]
print(f"✓ Yeni: {ok}, ⏭️  Atlandı: {skip}, ❌ Eksik NIfTI: {len(miss)}")
if miss: print("Eksik vakalar (ilk 5):", miss[:5])

# Unique label değerlerini raporla
_sample_masks = sorted((DATASET_DIR / "labelsTr").glob("*.nii.gz"))[:5]
if _sample_masks:
    _lbl_arr = sitk.GetArrayFromImage(sitk.ReadImage(str(_sample_masks[0])))
    print(f"\nÖrnek maske ({_sample_masks[0].name}) unique label'lar: {np.unique(_lbl_arr).tolist()}")


prep: 100%|██████████| 554/554 [00:00<00:00, 64434.71it/s]

✓ Yeni: 0, ⏭️  Atlandı: 553, ❌ Eksik NIfTI: 1
Eksik vakalar (ilk 5): ['missing:20317']

Örnek maske (case_20001.nii.gz) unique label'lar: [0, 2]


In [10]:
# Dataset dizini zaten dbf82eed'de tanımlandı:
# DATASET_DIR = NNUNET_RAW / f"Dataset{DATASET_ID}_{DATASET_NAME}"
print(f"Dataset dizini: {DATASET_DIR}")
print(f"  imagesTr : {len(list((DATASET_DIR/'imagesTr').glob('*.nii.gz')))} dosya")
print(f"  labelsTr : {len(list((DATASET_DIR/'labelsTr').glob('*.nii.gz')))} dosya")


Dataset dizini: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_raw/Dataset100_Abdomen
  imagesTr : 553 dosya
  labelsTr : 553 dosya


In [11]:
# nnU-Net v2 dataset.json formatı — channel_names + labels zorunlu
dataset_meta = {
    "channel_names": {"0": "CT"},
    "labels": {
        "background": 0,
        **{name: i + 1 for i, name in enumerate(SUPER_CLASSES)},
    },
    "numTraining": len(all_cases),
    "file_ending": ".nii.gz",
}
(DATASET_DIR / "dataset.json").write_text(json.dumps(dataset_meta, indent=2))
print("dataset.json yazıldı:", DATASET_DIR / "dataset.json")
print(json.dumps(dataset_meta, indent=2, ensure_ascii=False))


dataset.json yazıldı: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_raw/Dataset100_Abdomen/dataset.json
{
  "channel_names": {
    "0": "CT"
  },
  "labels": {
    "background": 0,
    "acute_cholecystitis": 1,
    "kidney_ureter_stone": 2,
    "acute_pancreatitis": 3,
    "aortic_aneurysm_dissection": 4,
    "acute_appendicitis": 5,
    "acute_diverticulitis": 6
  },
  "numTraining": 554,
  "file_ending": ".nii.gz"
}


## 5. nnU-Net v2 Plan + Preprocess

```bash
nnUNetv2_plan_and_preprocess -d 100 -c 3d_fullres --verify_dataset_integrity
```

Sonrasında `splits_final.json` bizim fold tanımımızla değiştirilir.

In [12]:
import shutil, sysconfig, json
import SimpleITK as sitk
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# ── DICOM → NIfTI (eksik dosyalar varsa) ─────────────────────────────────
fold_cases   = sorted(set(load_fold(fold, "train") + load_fold(fold, "val")))
train_cases  = load_fold(fold, "train")
val_cases    = load_fold(fold, "val")

_missing_nii = [c for c in fold_cases
                if not (NIFTI_DIR / f"case_{c:05d}_0000.nii.gz").exists()]
print(f"NIfTI: {len(fold_cases)-len(_missing_nii)} mevcut, {len(_missing_nii)} eksik")

if _missing_nii:
    print(f"DICOM → NIfTI dönüşümü ({len(_missing_nii)} vaka)...")

    def _dcm2nii(case: int):
        out = NIFTI_DIR / f"case_{case:05d}_0000.nii.gz"
        if out.exists(): return "skip"
        reader = sitk.ImageSeriesReader()
        names  = reader.GetGDCMSeriesFileNames(str(EGITIM_DIR / str(case)))
        if not names: return "no_dcm"
        reader.SetFileNames(names)
        try: sitk.WriteImage(reader.Execute(), str(out)); return "ok"
        except Exception as e: return f"err:{e}"

    with ThreadPoolExecutor(max_workers=4) as ex:
        _r = list(tqdm(ex.map(_dcm2nii, _missing_nii),
                        total=len(_missing_nii), desc="DICOM→NIfTI"))
    print(f"  ✓ {_r.count('ok')} yeni, ⏭ {_r.count('skip')} atlandı, "
          f"❌ {sum(1 for s in _r if s not in ('ok','skip'))} hata")

# ── Dataset dizinleri + dataset.json ─────────────────────────────────────
for _sub in ["imagesTr", "labelsTr"]:
    (DATASET_DIR / _sub).mkdir(parents=True, exist_ok=True)

_ds_json = DATASET_DIR / "dataset.json"
if not _ds_json.exists():
    _ds_json.write_text(json.dumps({
        "channel_names": {"0": "CT"},
        "labels": {"background": 0, **{n: i+1 for i, n in enumerate(SUPER_CLASSES)}},
        "numTraining": len(fold_cases), "file_ending": ".nii.gz",
    }, indent=2))
    print(f"dataset.json yazıldı")

# ── prep_case (imagesTr boşsa) ────────────────────────────────────────────
manifest = pd.read_csv(SPLIT_DIR / "manifest.csv")
_n_img = len(list((DATASET_DIR / "imagesTr").glob("*.nii.gz")))
print(f"imagesTr: {_n_img} dosya (hedef: {len(fold_cases)})")

if _n_img < len(fold_cases) // 2:
    print("prep_case çalıştırılıyor...")

    def _lc(src, dst):
        if dst.exists(): return
        try: os.symlink(src, dst)
        except (OSError, NotImplementedError): shutil.copy2(src, dst)

    def _prep(case):
        src = NIFTI_DIR / f"case_{case:05d}_0000.nii.gz"
        if not src.exists(): return f"miss"
        _lc(src, DATASET_DIR / "imagesTr" / f"case_{case:05d}_0000.nii.gz")
        dst_lbl = DATASET_DIR / "labelsTr" / f"case_{case:05d}.nii.gz"
        if dst_lbl.exists(): return "skip"
        boxes = lift_2d_to_3d(manifest, case)
        img_arr = sitk.GetArrayFromImage(sitk.ReadImage(str(src)))
        mask = np.zeros(img_arr.shape, dtype=np.uint8)
        for b in boxes:
            lbl = int(b["class"]) + 1
            z1,z2 = int(b["z1"]), min(int(b["z2"])+1, mask.shape[0])
            y1,y2 = int(b["y1"]), min(int(b["y2"])+1, mask.shape[1])
            x1,x2 = int(b["x1"]), min(int(b["x2"])+1, mask.shape[2])
            mask[z1:z2, y1:y2, x1:x2] = lbl
        m_itk = sitk.GetImageFromArray(mask)
        m_itk.CopyInformation(sitk.ReadImage(str(src)))
        sitk.WriteImage(m_itk, str(dst_lbl))
        return "ok"

    _res = [_prep(c) for c in tqdm(fold_cases, desc="prep")]
    print(f"  ✓ {_res.count('ok')} yeni, ⏭ {_res.count('skip')} atlandı, "
          f"❌ {sum(1 for r in _res if r=='miss')} eksik NIfTI")

_n_img = len(list((DATASET_DIR / "imagesTr").glob("*.nii.gz")))
if _n_img == 0:
    raise RuntimeError(f"imagesTr boş! EGITIM_DIR var mı? {EGITIM_DIR.exists()}")
print(f"\nDataset hazır: imagesTr={_n_img}, "
      f"labelsTr={len(list((DATASET_DIR/'labelsTr').glob('*.nii.gz')))}")

# ── nnUNetv2_plan_and_preprocess ──────────────────────────────────────────
def _nnunet_cmd(name: str) -> str:
    p = shutil.which(name)
    if p: return p
    for d in [sysconfig.get_path("scripts"), str(Path(sys.executable).parent),
               "/usr/local/bin", "/opt/conda/bin"]:
        c = Path(d) / name
        if c.exists(): return str(c)
    return name  # fallback: hope it's on PATH

_env = {**os.environ}
_cmd = _nnunet_cmd("nnUNetv2_plan_and_preprocess")
print(f"\nÇalıştırılıyor: {_cmd} -d {DATASET_ID} -c 3d_fullres --verify_dataset_integrity")
r = subprocess.run(
    [_cmd, "-d", str(DATASET_ID), "-c", "3d_fullres", "--verify_dataset_integrity"],
    env=_env, capture_output=True, text=True, timeout=3600
)
if r.returncode != 0:
    print("STDERR:\n", r.stderr[-2000:])
    print("✗ plan_and_preprocess başarısız!")
else:
    print("✓ plan_and_preprocess tamamlandı!")

# ── Özel fold split yaz (nnU-Net'in auto-split'ini geçersiz kıl) ─────────
_splits_dir = NNUNET_PREPROCESSED / f"Dataset{DATASET_ID}_{DATASET_NAME}"
_splits_dir.mkdir(parents=True, exist_ok=True)
_splits_path = _splits_dir / "splits_final.json"

# nnU-Net v2 splits formatı: her fold için case identifier listesi
# Case identifier = dosya adından _0000.nii.gz çıkarılmış hali
_splits = [{
    "train": [f"case_{c:05d}" for c in train_cases],
    "val":   [f"case_{c:05d}" for c in val_cases],
}]
_splits_path.write_text(json.dumps(_splits, indent=2))
print(f"\nÖzel split yazıldı: {_splits_path}")
print(f"  train: {len(train_cases)} vaka, val: {len(val_cases)} vaka")


NIfTI: 553 mevcut, 1 eksik
DICOM → NIfTI dönüşümü (1 vaka)...


DICOM→NIfTI: 100%|██████████| 1/1 [00:00<00:00,  7.71it/s]

  ✓ 0 yeni, ⏭ 0 atlandı, ❌ 1 hata
imagesTr: 553 dosya (hedef: 554)

Dataset hazır: imagesTr=553, labelsTr=553

Çalıştırılıyor: /Users/ramazanpolat/Desktop/datasets/abdomen/.venv/bin/nnUNetv2_plan_and_preprocess -d 100 -c 3d_fullres --verify_dataset_integrity


TimeoutExpired: Command '['/Users/ramazanpolat/Desktop/datasets/abdomen/.venv/bin/nnUNetv2_plan_and_preprocess', '-d', '100', '-c', '3d_fullres', '--verify_dataset_integrity']' timed out after 3600 seconds

In [17]:
import shutil, os, json

# ── Split dosyalarını yeniden yükle (20003 kaldırıldı) ─────────────────────
print("Split dosyaları güncellendi: case 20003 kaldırıldı\n")

train_cases = load_fold(fold, "train")
val_cases   = load_fold(fold, "val")
all_cases   = sorted(set(train_cases + val_cases))

print(f"Fold {fold}:")
print(f"  Train: {len(train_cases)} vaka")
print(f"  Val:   {len(val_cases)} vaka")
print(f"  Total: {len(all_cases)} vaka")

# ── NIfTI dosyalarını kontrol et ────────────────────────────────────────────
_nii_files = sorted(NIFTI_DIR.glob("*.nii.gz"))
print(f"\nNIfTI dosyaları: {len(_nii_files)}")

# Tüm NIfTI'leri imagesTr'ye link et
_linked = 0
for _nii in _nii_files:
    _dst = DATASET_DIR / "imagesTr" / _nii.name
    if not _dst.exists():
        try:
            os.symlink(_nii, _dst)
            _linked += 1
        except:
            shutil.copy2(_nii, _dst)
            _linked += 1

_n_img = len(list((DATASET_DIR / "imagesTr").glob("*.nii.gz")))
_n_lbl = len(list((DATASET_DIR / "labelsTr").glob("*.nii.gz")))
print(f"\nDataset hazır:")
print(f"  imagesTr: {_n_img} dosya (link: +{_linked})")
print(f"  labelsTr: {_n_lbl} dosya")

if _n_img != _n_lbl:
    print(f"⚠️  UYARI: imagesTr ({_n_img}) ≠ labelsTr ({_n_lbl})")
    # Missing labels'ları oluştur
    _missing = 0
    manifest = pd.read_csv(SPLIT_DIR / "manifest.csv")
    import SimpleITK as sitk
    import numpy as np
    
    for _case in all_cases:
        _dst_lbl = DATASET_DIR / "labelsTr" / f"case_{_case:05d}.nii.gz"
        if not _dst_lbl.exists():
            boxes_3d = lift_2d_to_3d(manifest, _case)
            src_nii = DATASET_DIR / "imagesTr" / f"case_{_case:05d}_0000.nii.gz"
            if src_nii.exists():
                img_itk = sitk.ReadImage(str(src_nii))
                img_arr = sitk.GetArrayFromImage(img_itk)
                mask_arr = np.zeros(img_arr.shape, dtype=np.uint8)
                
                for b in boxes_3d:
                    label = int(b["class"]) + 1
                    z1 = int(b["z1"]); z2 = min(int(b["z2"])+1, mask_arr.shape[0])
                    y1 = int(b["y1"]); y2 = min(int(b["y2"])+1, mask_arr.shape[1])
                    x1 = int(b["x1"]); x2 = min(int(b["x2"])+1, mask_arr.shape[2])
                    mask_arr[z1:z2, y1:y2, x1:x2] = label
                
                mask_itk = sitk.GetImageFromArray(mask_arr)
                mask_itk.CopyInformation(img_itk)
                sitk.WriteImage(mask_itk, str(_dst_lbl))
                _missing += 1
    
    print(f"  → {_missing} eksik label oluşturuldu")
    _n_lbl = len(list((DATASET_DIR / "labelsTr").glob("*.nii.gz")))
    print(f"  → labelsTr: {_n_lbl} dosya")

# ── nnUNetv2_plan_and_preprocess (temiz çalıştırma) ───────────────────────
print(f"\nnn UNet v2 plan_and_preprocess çalıştırılıyor...")

# Preprocessed klasörünü temizle
if NNUNET_PREPROCESSED.exists():
    print(f"Temizleniyor: {NNUNET_PREPROCESSED}")
    shutil.rmtree(NNUNET_PREPROCESSED)

# Çalıştır
_cmd = _nnunet_cmd("nnUNetv2_plan_and_preprocess")
r = subprocess.run(
    [_cmd, "-d", str(DATASET_ID), "-c", "3d_fullres", "--verify_dataset_integrity"],
    env={**os.environ}, capture_output=True, text=True, timeout=3600
)

if r.returncode == 0 or "Fingerprint extraction" in r.stdout:
    print("✓ plan_and_preprocess başarılı")
else:
    print("❌ Hata:")
    print(r.stdout[-1000:])
    if r.stderr:
        print(r.stderr[-500:])

# ── splits_final.json yaz ──────────────────────────────────────────────────
_splits_path = NNUNET_PREPROCESSED / f"Dataset{DATASET_ID}_{DATASET_NAME}" / "splits_final.json"
_splits_path.parent.mkdir(parents=True, exist_ok=True)

_splits = [{
    "train": [f"case_{c:05d}" for c in train_cases],
    "val":   [f"case_{c:05d}" for c in val_cases],
}]
_splits_path.write_text(json.dumps(_splits, indent=2))
print(f"\n✓ splits_final.json yazıldı: {_splits_path}")

Split dosyaları güncellendi: case 20003 kaldırıldı

Fold 0:
  Train: 443 vaka
  Val:   111 vaka
  Total: 554 vaka

NIfTI dosyaları: 553

Dataset hazır:
  imagesTr: 553 dosya (link: +0)
  labelsTr: 553 dosya

nn UNet v2 plan_and_preprocess çalıştırılıyor...
Temizleniyor: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_preprocessed
✓ plan_and_preprocess başarılı

✓ splits_final.json yazıldı: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_preprocessed/Dataset100_Abdomen/splits_final.json


## 6. Fold 0 Eğitim

In [ ]:
# ── Preprocessed veriyi Drive'dan Colab local diske kopyala ───────────────
# Google Drive FUSE üzerinden çok-thread'li NPZ okuma bağlantıyı koparıyor.
# Çözüm: eğitim başlamadan önce preprocessed klasörü /content/ altına al.
import shutil, time

LOCAL_PREPROCESSED = Path('/content/nnunet_preprocessed')

if IS_COLAB:
    drive_pre   = NNUNET_PREPROCESSED
    dataset_sub = f'Dataset{DATASET_ID}_{DATASET_NAME}'
    local_sub   = LOCAL_PREPROCESSED / dataset_sub

    if not local_sub.exists():
        src = drive_pre / dataset_sub
        if not src.exists():
            raise FileNotFoundError(
                f'Preprocessed klasörü bulunamadı: {src}\n'
                'Önce plan_and_preprocess hücresini çalıştırın.'
            )
        print(f'Preprocessed verisi kopyalanıyor: {src}')
        print(f'  → {local_sub}')
        t0 = time.time()
        shutil.copytree(str(src), str(local_sub))
        print(f'  ✓ Tamamlandı ({time.time()-t0:.0f}s)')
    else:
        print(f'Local preprocessed zaten mevcut: {local_sub}')

    # Ortam değişkenini yerel yola güncelle
    os.environ['nnUNet_preprocessed'] = str(LOCAL_PREPROCESSED)
    NNUNET_PREPROCESSED = LOCAL_PREPROCESSED
    print(f'nnUNet_preprocessed → {os.environ["nnUNet_preprocessed"]}')
else:
    print('Colab dışı ortam — kopyalama atlandı.')


In [ ]:
import torch

# ── GPU kontrolü ──────────────────────────────────────────────────────────
_has_cuda = torch.cuda.is_available()
_device   = "cuda" if _has_cuda else "cpu"
print(f"Cihaz: {_device}")
if _has_cuda:
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

if not _has_cuda:
    print("\n⚠️  CUDA bulunamadı — eğitim GPU gerektirir.")
    print("Bu notebook Colab'da GPU runtime ile çalıştırılmalıdır:")
    print("  Runtime → Change runtime type → T4 GPU (veya A100)")
    raise SystemExit("GPU olmadan eğitim başlatılmadı.")

# ── nnUNetv2_train ────────────────────────────────────────────────────────
print(f"\nnnU-Net v2 eğitimi — fold 0, 3d_fullres, ~8-15 saat (T4)...")
print(f"Çıktılar: {NNUNET_RESULTS}")

_cmd = _nnunet_cmd("nnUNetv2_train")
# --npz: softmax olasılık haritalarını kaydeder (güven skoru için)
r = subprocess.run(
    [_cmd, str(DATASET_ID), "3d_fullres", "0", "--npz"],
    env={**os.environ}, capture_output=False, text=True
)
print("Eğitim çıktı kodu:", r.returncode)
if r.returncode != 0:
    print("⚠️  Eğitim hata kodu döndürüyor (bkz. yukarı)")


## 7. 3B Çıkarım

In [ ]:
import shutil, sysconfig, os

# Validasyon seti görüntüleri ayrı predict input dizinine (nnunet_raw DıŞINDA)
# nnU-Net Dataset ID çakışmasından kaçınmak için nnunet_raw dışında tut
VAL_INPUT_DIR  = NND_ROOT / "val_input_raw"
VAL_OUTPUT_DIR = NNUNET_RESULTS / f"Dataset{DATASET_ID}_{DATASET_NAME}" / "nnUNetTrainer__nnUNetPlans__3d_fullres" / "fold_0" / "val_predictions"
VAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
VAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Val görüntülerini input dizinine semlink (nnunet_raw DIŞINDAKİ dizine)
_linked = 0
for _c in val_cases:
    _src = DATASET_DIR / "imagesTr" / f"case_{_c:05d}_0000.nii.gz"
    _dst = VAL_INPUT_DIR / f"case_{_c:05d}_0000.nii.gz"
    if _src.exists() and not _dst.exists():
        try:
            os.symlink(_src, _dst)
            _linked += 1
        except:
            shutil.copy2(_src, _dst)
            _linked += 1

print(f"Val input dizini (nnunet_raw dışında): {VAL_INPUT_DIR}")
print(f"  → {len(list(VAL_INPUT_DIR.glob('*.nii.gz')))} dosya ({_linked} yeni link/kopya)")

# nnUNetv2_predict
_cmd = _nnunet_cmd("nnUNetv2_predict")
print(f"\nTahmin başlatılıyor: {_cmd}")
r = subprocess.run([
    _cmd,
    "-i", str(VAL_INPUT_DIR),
    "-o", str(VAL_OUTPUT_DIR),
    "-d", str(DATASET_ID),
    "-c", "3d_fullres",
    "-f", "0",
    "--save_probabilities",   # .npz olasılık haritaları (güven skoru için)
], env={**os.environ}, capture_output=True, text=True, timeout=3600*4)

print("STDOUT:\n", r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR:\n", r.stderr[-1000:])
else:
    _preds = list(VAL_OUTPUT_DIR.glob("*.nii.gz"))
    print(f"✓ Tahmin tamamlandı: {len(_preds)} NIfTI mask")

Val input dizini: 111 dosya
Tahmin başlatılıyor: /Users/ramazanpolat/Desktop/datasets/abdomen/.venv/bin/nnUNetv2_predict
STDOUT:
 
#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################


STDERR:
 ties.py", line 22, in get_output_folder
    tmp = join(nnUNet_results, maybe_convert_to_dataset_name(dataset_name_or_id),
  File "/Users/ramazanpolat/Desktop/datasets/abdomen/.venv/lib/python3.9/site-packages/nnunetv2/utilities/dataset_name_id_conversion.py", line 74, in maybe_convert_to_dataset_name
    return convert_id_to_dataset_name(dataset_name_or_id)
  File "/Users/ramazanpolat/Desktop/datasets/abdomen/.venv/lib/python3.9/site-packages/nnunetv2/u

In [20]:
import numpy as np
import SimpleITK as sitk
from scipy import ndimage
from PIL import Image

PRED_DIR = VAL_OUTPUT_DIR
print("Tahmin dizini:", PRED_DIR)
print(f"→ {len(list(PRED_DIR.glob('*.nii.gz')))} NIfTI mask")

def seg_to_bboxes_2d(pred_nii_path: Path) -> list:
    """
    nnU-Net v2 semantic segmentation mask → 2D-projeksiyon bounding box satırları.

    Her bağlı bileşen (connected component) bir lezyon örneği = bir 3B bbox.
    3B bbox kapsadığı her z-kesitine yansıtılır (nnDetection ile aynı projeksiyon).
    Güven skoru: bileşenin voksel hacmi / tüm o sınıf voksel sayısı.
    .npz olasılık dosyası varsa max softmax olasılığı kullanılır.
    """
    # Case numarası: case_20001.nii.gz → 20001
    try:
        case = int(pred_nii_path.stem.split("_")[1])
    except (IndexError, ValueError):
        return []

    mask = sitk.GetArrayFromImage(sitk.ReadImage(str(pred_nii_path)))  # [Z,Y,X]

    # Olasılık haritası varsa yükle (softmax)
    prob_path = pred_nii_path.with_suffix("").with_suffix(".npz")
    probs = None
    if prob_path.exists():
        probs = np.load(prob_path)["probabilities"]  # [C, Z, Y, X]

    rows = []
    for label_id in range(1, len(SUPER_CLASSES) + 1):
        binary = (mask == label_id).astype(np.uint8)
        if binary.sum() == 0:
            continue
        labeled, n_comp = ndimage.label(binary)
        total_vox = float(binary.sum())

        for comp_id in range(1, n_comp + 1):
            comp_mask = (labeled == comp_id)
            coords = np.where(comp_mask)
            z1, z2 = int(coords[0].min()), int(coords[0].max())
            y1, y2 = int(coords[1].min()), int(coords[1].max())
            x1, x2 = int(coords[2].min()), int(coords[2].max())

            # Güven skoru: softmax prob > hacim bazlı
            if probs is not None:
                score = float(probs[label_id][comp_mask].mean())
            else:
                score = float(comp_mask.sum()) / total_vox

            # 3B bbox'ı kapsadığı her z-kesitine yansıt
            for z in range(z1, z2 + 1):
                rows.append({
                    "case": case, "image_id": z,
                    "class": label_id - 1,   # 0-indexed sınıf
                    "score": score,
                    "x1": float(x1), "y1": float(y1),
                    "x2": float(x2), "y2": float(y2),
                })
    return rows

all_rows = []
for _p in sorted(PRED_DIR.glob("*.nii.gz")):
    all_rows.extend(seg_to_bboxes_2d(_p))

pred_df = pd.DataFrame(all_rows)
print(f"\n3B→2D projeksiyon: {len(pred_df):,} satır, "
      f"{pred_df['case'].nunique() if not pred_df.empty else 0} vaka")
if not pred_df.empty:
    print(pred_df.groupby("class")["score"].describe().round(3))


Tahmin dizini: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_results/Dataset100_Abdomen/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/val_predictions
→ 0 NIfTI mask

3B→2D projeksiyon: 0 satır, 0 vaka


In [21]:
from src.evaluation import top5_f1_mean, f1_at_iou

val_lbl_dir = DET_DATA_DIR / f"fold{fold}" / "labels" / "val"
val_img_dir = DET_DATA_DIR / f"fold{fold}" / "images" / "val"

gt_rows = []
for lp in val_lbl_dir.glob("*.txt"):
    ip = val_img_dir / (lp.stem + ".png")
    if not ip.exists():
        continue
    W, H = Image.open(ip).size
    stem = lp.stem
    case, image_id = (stem.split("_", 1) + ["0"])[:2]
    for line in lp.read_text().strip().splitlines():
        p = line.split()
        if len(p) < 5:
            continue
        cl = int(p[0]); cx, cy, w, h = map(float, p[1:5])
        gt_rows.append({
            "case": int(case), "image_id": int(image_id), "class": cl,
            "x1": (cx - w / 2) * W, "y1": (cy - h / 2) * H,
            "x2": (cx + w / 2) * W, "y2": (cy + h / 2) * H,
        })

gt_df = pd.DataFrame(gt_rows)
print(f"GT bbox sayısı: {len(gt_df):,}")

if not pred_df.empty:
    result = top5_f1_mean(pred_df, gt_df)
    detail = f1_at_iou(pred_df, gt_df, iou_th=0.3)
    print(f"\nnnDetection fold {fold} — Top-5 F1 : {result['top5_mean_f1']:.4f}")
    print(f"Macro F1 @ IoU 0.3                : {detail['macro_f1']:.4f}")
    print(f"Micro F1 @ IoU 0.3                : {detail['micro_f1']:.4f}")
else:
    print("Tahmin yok — nndet_predict + nndet_consolidate adımlarının bitmesini bekleyin.")

GT bbox sayısı: 5,678
Tahmin yok — nndet_predict + nndet_consolidate adımlarının bitmesini bekleyin.


## 8. Faz 4 Çıktı Özeti

| Çıktı | Yol |
|---|---|
| NIfTI hacimler | `nndet/nifti/case_*.nii.gz` |
| nnU-Net v2 ham veri | `nndet/nnunet_raw/Dataset100_Abdomen/` |
| Preprocessed | `nndet/nnunet_preprocessed/Dataset100_Abdomen/` |
| Eğitilmiş model | `nndet/nnunet_results/Dataset100_Abdomen/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/` |
| Tahmin maskeleri | `.../fold_0/val_predictions/*.nii.gz` |
| Olasılık haritaları | `.../fold_0/val_predictions/*.npz` |

**Seg → bbox pipeline:**
Predicted NIfTI mask → her sınıf için binary mask → connected components → 3B bbox → z-ekseni projeksiyon → 2D F1@IoU

**Sonraki:** `Faz5_Harici_Test_Yarisma.ipynb`